# 🛒 Superstore Sales — Exploratory Data Analysis
**Author:** Your Name  
**Dataset:** Sample Superstore (9,994 orders, 2014–2017)  
**Goal:** Uncover profitability drivers, diagnose losses, and recommend business actions.

---
## Table of Contents
1. [Setup & Data Loading](#1-setup)
2. [Data Overview & Cleaning](#2-overview)
3. [Sales & Profit Trends Over Time](#3-trends)
4. [Category & Sub-Category Performance](#4-categories)
5. [The Discount Problem](#5-discounts)
6. [Regional Performance](#6-regions)
7. [Customer Segment Analysis](#7-segments)
8. [Key Findings & Business Recommendations](#8-recommendations)

## 1. Setup & Data Loading <a id='1-setup'></a>

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'font.family': 'sans-serif',
})
PALETTE = ['#2563EB','#16A34A','#DC2626','#D97706','#7C3AED','#0891B2']
sns.set_palette(PALETTE)

print('Libraries loaded ✓')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
# Update this path to wherever you saved the CSV
df = pd.read_csv('Sample_-_Superstore.csv', encoding='latin1')

# Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
df.head()

## 2. Data Overview & Cleaning <a id='2-overview'></a>

In [ ]:
# ── Data quality check ────────────────────────────────────────────────────────
print('=== Missing values ===')
print(df.isnull().sum())

print('\n=== Data types ===')
print(df.dtypes)

print('\n=== Numeric summary ===')
df[['Sales','Quantity','Discount','Profit']].describe().round(2)

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
df['Year']          = df['Order Date'].dt.year
df['Month']         = df['Order Date'].dt.to_period('M')
df['Profit Margin'] = df['Profit'] / df['Sales']          # profit as % of sales
df['Days to Ship']  = (df['Ship Date'] - df['Order Date']).dt.days

# Discount buckets for grouping
df['Discount Bucket'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
    labels=['0%', '1–10%', '11–20%', '21–30%', '31–50%', '51%+']
)

print('New columns added: Year, Month, Profit Margin, Days to Ship, Discount Bucket ✓')

# Key KPIs
print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━
Total Sales   : ${df['Sales'].sum():>12,.0f}
Total Profit  : ${df['Profit'].sum():>12,.0f}
Profit Margin :  {df['Profit'].sum()/df['Sales'].sum()*100:>10.1f}%
Unique Orders :  {df['Order ID'].nunique():>10,}
Unique Customers: {df['Customer ID'].nunique():>8,}
━━━━━━━━━━━━━━━━━━━━━━━━
""")

## 3. Sales & Profit Trends Over Time <a id='3-trends'></a>
**Question:** Is the business growing? Are sales and profit moving together?

In [ ]:
# ── Annual trend ──────────────────────────────────────────────────────────────
yearly = df.groupby('Year')[['Sales','Profit']].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sales bar chart
bars = axes[0].bar(yearly['Year'], yearly['Sales'], color=PALETTE[0], width=0.5)
axes[0].set_title('Annual Sales')
axes[0].set_ylabel('Sales ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
for bar, val in zip(bars, yearly['Sales']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5000,
                 f'${val/1e3:.0f}K', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Profit bar chart
bars2 = axes[1].bar(yearly['Year'], yearly['Profit'], color=PALETTE[1], width=0.5)
axes[1].set_title('Annual Profit')
axes[1].set_ylabel('Profit ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
for bar, val in zip(bars2, yearly['Profit']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                 f'${val/1e3:.0f}K', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_annual_trend.png', bbox_inches='tight')
plt.show()
print('\n💡 Insight: Both sales and profit grew consistently. Sales grew 51% from 2014→2017.')

In [ ]:
# ── Monthly trend (seasonality) ───────────────────────────────────────────────
monthly = df.groupby('Month')[['Sales','Profit']].sum().reset_index()
monthly['Month_str'] = monthly['Month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly['Month_str'], monthly['Sales'],   color=PALETTE[0], linewidth=2,   label='Sales')
ax.plot(monthly['Month_str'], monthly['Profit'],  color=PALETTE[1], linewidth=2,   label='Profit')
ax.fill_between(monthly['Month_str'], monthly['Profit'], alpha=0.15, color=PALETTE[1])
ax.set_title('Monthly Sales & Profit (2014–2017)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
ax.set_xticks(ax.get_xticks()[::6])   # show every 6th label to avoid crowding
ax.tick_params(axis='x', rotation=45)
ax.legend()
ax.axhline(0, color='red', linewidth=0.8, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('chart_monthly_trend.png', bbox_inches='tight')
plt.show()
print('\n💡 Insight: Q4 spikes in sales every year (holiday season), but profit spikes are smaller — suggesting heavier discounting in Q4.')

## 4. Category & Sub-Category Performance <a id='4-categories'></a>
**Question:** Which products make money and which are bleeding cash?

In [ ]:
# ── Category-level profit ─────────────────────────────────────────────────────
cat = df.groupby('Category')[['Sales','Profit']].sum().reset_index()
cat['Margin'] = cat['Profit'] / cat['Sales'] * 100

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for i, (col, label, fmt) in enumerate([
    ('Sales',  'Total Sales ($)',  lambda x,_: f'${x/1e3:.0f}K'),
    ('Profit', 'Total Profit ($)', lambda x,_: f'${x/1e3:.0f}K'),
    ('Margin', 'Profit Margin (%)',lambda x,_: f'{x:.0f}%'),
]):
    colors = [PALETTE[2] if v < 0 else PALETTE[i] for v in cat[col]]
    axes[i].bar(cat['Category'], cat[col], color=colors, width=0.5)
    axes[i].set_title(label)
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
    axes[i].tick_params(axis='x', rotation=10)

plt.suptitle('Performance by Category', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart_category.png', bbox_inches='tight')
plt.show()
print('\n💡 Insight: Technology has the highest profit margin. Furniture sells a lot but earns the least.')

In [ ]:
# ── Sub-category profit waterfall ─────────────────────────────────────────────
subcat = df.groupby('Sub-Category')['Profit'].sum().sort_values()

colors = ['#DC2626' if x < 0 else '#16A34A' for x in subcat]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(subcat.index, subcat.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Total Profit ($)')
ax.set_title('Profit by Sub-Category  (red = loss-making)', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))

# Add value labels
for bar, val in zip(bars, subcat.values):
    xpos = val + 300 if val >= 0 else val - 300
    ha   = 'left'   if val >= 0 else 'right'
    ax.text(xpos, bar.get_y()+bar.get_height()/2,
            f'${val:,.0f}', va='center', ha=ha, fontsize=8)

plt.tight_layout()
plt.savefig('chart_subcat_profit.png', bbox_inches='tight')
plt.show()
print('\n💡 Insight: Tables LOSE $17,725. Copiers, Phones, and Accessories are the profit engines.')

## 5. The Discount Problem <a id='5-discounts'></a>
**Question:** Are discounts hurting profitability? This is the most important finding in this dataset.

In [ ]:
# ── Profit by discount bucket ──────────────────────────────────────────────────
disc = df.groupby('Discount Bucket', observed=True)[['Profit','Sales']].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#DC2626' if v < 0 else '#2563EB' for v in disc['Profit']]
axes[0].bar(disc['Discount Bucket'], disc['Profit'], color=colors)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Avg Profit per Order by Discount Level')
axes[0].set_xlabel('Discount')
axes[0].set_ylabel('Avg Profit ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.0f}'))
for i, (col, val) in enumerate(zip(disc['Discount Bucket'], disc['Profit'])):
    axes[0].text(i, val + (3 if val >= 0 else -8), f'${val:.0f}', ha='center', fontsize=9)

# Scatter: discount vs profit (sample 2000 rows for clarity)
sample = df.sample(2000, random_state=42)
axes[1].scatter(sample['Discount'], sample['Profit'], alpha=0.3, s=15, color=PALETTE[0])
# Trend line
z = np.polyfit(df['Discount'], df['Profit'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 0.8, 100)
axes[1].plot(x_line, p(x_line), color='red', linewidth=2, label=f'Trend')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Discount vs Profit (scatter + trend)')
axes[1].set_xlabel('Discount Rate')
axes[1].set_ylabel('Profit ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0%}'))
axes[1].legend()

plt.tight_layout()
plt.savefig('chart_discount.png', bbox_inches='tight')
plt.show()

# Correlation
corr = df['Discount'].corr(df['Profit'])
print(f'\n💡 Correlation between Discount and Profit: {corr:.3f}')
print('   Orders with >20% discount are, on average, UNPROFITABLE.')
print('   31–50% discount buckets average a LOSS of $156 per order!')

In [ ]:
# ── Which sub-categories are being over-discounted? ────────────────────────────
disc_subcat = df.groupby('Sub-Category').agg(
    Avg_Discount=('Discount','mean'),
    Total_Profit=('Profit','sum')
).reset_index().sort_values('Avg_Discount', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#DC2626' if v < 0 else '#2563EB' for v in disc_subcat['Total_Profit']]
bars = ax.bar(disc_subcat['Sub-Category'], disc_subcat['Avg_Discount']*100, color=colors)
ax.set_title('Average Discount % by Sub-Category  (red = loss-making)', fontweight='bold')
ax.set_ylabel('Average Discount (%)')
ax.tick_params(axis='x', rotation=45)
ax.axhline(20, color='orange', linewidth=1.5, linestyle='--', label='20% danger zone')
ax.legend()
for bar, val in zip(bars, disc_subcat['Avg_Discount']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val:.0%}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('chart_discount_subcat.png', bbox_inches='tight')
plt.show()
print('\n💡 Insight: Tables and Binders receive the heaviest discounts AND are loss-making. Coincidence? No.')

## 6. Regional Performance <a id='6-regions'></a>
**Question:** Which regions should the business invest in vs. fix?

In [ ]:
# ── Region summary ────────────────────────────────────────────────────────────
region = df.groupby('Region').agg(
    Sales=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','count')
).reset_index()
region['Margin'] = region['Profit'] / region['Sales'] * 100
region = region.sort_values('Profit', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sales vs Profit grouped bars
x = range(len(region))
width = 0.35
axes[0].bar([i - width/2 for i in x], region['Sales'],  width, label='Sales',  color=PALETTE[0])
axes[0].bar([i + width/2 for i in x], region['Profit'], width, label='Profit', color=PALETTE[1])
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(region['Region'])
axes[0].set_title('Sales vs Profit by Region')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1e3:.0f}K'))
axes[0].legend()

# Profit margin bars
colors = [PALETTE[1] if v > 10 else PALETTE[3] for v in region['Margin']]
axes[1].bar(region['Region'], region['Margin'], color=colors)
axes[1].set_title('Profit Margin % by Region')
axes[1].set_ylabel('Margin (%)')
for i, (reg, val) in enumerate(zip(region['Region'], region['Margin'])):
    axes[1].text(i, val+0.2, f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_regions.png', bbox_inches='tight')
plt.show()
print('\n💡 Insight: West is the star region (14.9% margin). Central has the lowest margin — worth investigating why.')

In [ ]:
# ── Top 10 states by profit ────────────────────────────────────────────────────
state_profit = df.groupby('State')['Profit'].sum().sort_values()
bottom5 = state_profit.head(5)
top5    = state_profit.tail(5)
display_states = pd.concat([bottom5, top5])

colors = ['#DC2626' if v < 0 else '#16A34A' for v in display_states]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(display_states.index, display_states.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 5 & Bottom 5 States by Profit', fontweight='bold')
ax.set_xlabel('Total Profit ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
for i, val in enumerate(display_states.values):
    xpos = val + 200 if val >= 0 else val - 200
    ha   = 'left'   if val >= 0 else 'right'
    ax.text(xpos, i, f'${val:,.0f}', va='center', ha=ha, fontsize=9)
plt.tight_layout()
plt.savefig('chart_states.png', bbox_inches='tight')
plt.show()
print('\n💡 Insight: Texas and Ohio are significantly loss-making despite high sales volume. Discount abuse likely.')

## 7. Customer Segment Analysis <a id='7-segments'></a>
**Question:** Which customer type is the most valuable?

In [ ]:
# ── Segment performance ───────────────────────────────────────────────────────
seg = df.groupby('Segment').agg(
    Sales   =('Sales','sum'),
    Profit  =('Profit','sum'),
    Orders  =('Order ID','count'),
    Customers=('Customer ID','nunique')
).reset_index()
seg['Avg Order Value'] = seg['Sales'] / seg['Orders']
seg['Margin'] = seg['Profit'] / seg['Sales'] * 100

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Pie chart — share of sales
axes[0].pie(seg['Sales'], labels=seg['Segment'], autopct='%1.1f%%',
            colors=PALETTE[:3], startangle=90)
axes[0].set_title('Sales Share by Segment')

# Margin comparison
axes[1].bar(seg['Segment'], seg['Margin'], color=PALETTE[:3])
axes[1].set_title('Profit Margin % by Segment')
axes[1].set_ylabel('Margin (%)')
for i, val in enumerate(seg['Margin']):
    axes[1].text(i, val+0.1, f'{val:.1f}%', ha='center', fontweight='bold')

# Average order value
axes[2].bar(seg['Segment'], seg['Avg Order Value'], color=PALETTE[:3])
axes[2].set_title('Avg Order Value by Segment')
axes[2].set_ylabel('Avg Order ($)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.0f}'))
for i, val in enumerate(seg['Avg Order Value']):
    axes[2].text(i, val+1, f'${val:.0f}', ha='center', fontweight='bold')

plt.suptitle('Customer Segment Analysis', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart_segments.png', bbox_inches='tight')
plt.show()
print(seg[['Segment','Sales','Profit','Margin','Avg Order Value']].to_string(index=False))
print('\n💡 Insight: Home Office has fewer orders but a strong margin. Consumer drives volume.')

## 8. Key Findings & Business Recommendations <a id='8-recommendations'></a>

In [ ]:
# Summary stats for the write-up
total_loss_tables = df[df['Sub-Category']=='Tables']['Profit'].sum()
high_disc_orders  = (df['Discount'] > 0.2).sum()
high_disc_pct     = high_disc_orders / len(df) * 100
high_disc_profit  = df[df['Discount'] > 0.2]['Profit'].sum()

print("=" * 60)
print("   KEY FINDINGS & RECOMMENDATIONS")
print("=" * 60)

print("\nFINDING 1: Discounts above 20% destroy profit")
print("-" * 45)
print(f"  Correlation (discount vs profit): {df['Discount'].corr(df['Profit']):.3f}")
print(f"  {high_disc_pct:.0f}% of orders have a discount over 20%")
print(f"  Those orders lost a combined: ${abs(high_disc_profit):,.0f}")
print("  --> Recommend: Cap discounts at 20% across all categories.")

print("\nFINDING 2: Tables sub-category is losing money")
print("-" * 45)
print(f"  Tables lost ${abs(total_loss_tables):,.0f} over 4 years on $206K in sales")
print("  --> Recommend: Renegotiate supplier cost or discontinue.")

print("\nFINDING 3: West region leads in profit margin")
print("-" * 45)
print("  West: 14.9% margin vs Central's 7.9%")
print("  --> Recommend: Replicate West pricing strategy in Central.")

print("\nFINDING 4: Technology is under-invested")
print("-" * 45)
print("  Copiers: $55K profit on $149K sales (37% margin!)")
print("  --> Recommend: Increase Technology marketing budget.")
print("\n" + "=" * 60)


---
## Next Steps (Portfolio Upgrades)
- **Phase 3:** Build an interactive Tableau Public dashboard with these same charts
- **Phase 4:** Add customer segmentation using K-Means clustering (RFM analysis)
- **Phase 4:** Build a sales forecasting model using Facebook Prophet
- **Phase 5:** Push to GitHub with a clean README and share on LinkedIn

---
*Analysis by [Your Name] · Dataset: Tableau Sample Superstore*